# Entraînement YOLOv8 — conditions de peau (Phase 2 Lumina)

Exporte `best.onnx` pour `SKIN_YOLO_MODEL_PATH=models/best.onnx`.

## Amélioration 5 classes (garder blackheads)

1. **Step 0** — `yolov8s.pt` + `IMGSZ=1024` + `EPOCHS=200` ; logs ≈ **11M** params (pas 3M).
2. **Step 1** — Roboflow : Health Check, réparer **blackheads**, ≥300 images / ≥500 boxes par classe, nouvelle version + aug.
3. **Step 2** — `VERSION` = nouvelle version, Run all.
4. **Step 3** — `results.png`, `confusion_matrix.png`, `PR_curve.png`.
5. **Export** — `best.onnx` → `backend/models/best.onnx`, restart backend.

| id | classe |
|----|--------|
| 0 | acne |
| 1 | blackheads |
| 2 | dark_spots |
| 3 | enlarged_pores |
| 4 | wrinkles |

**Avant :** Runtime → Change runtime type → **T4 GPU**. Clé API : Colab Secret `ROBOFLOW_API_KEY` (ou saisie).

In [ ]:
!nvidia-smi -L

In [ ]:
%pip install -q ultralytics roboflow pyyaml

In [ ]:
# Configuration — Step 0 / Step 2
import os

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "").strip()

if not ROBOFLOW_API_KEY:
    from getpass import getpass
    ROBOFLOW_API_KEY = getpass("Roboflow API key: ").strip()

WORKSPACE = "hamoudas-workspace"
PROJECT = "face-skin-problems-e7y0m"
VERSION = 2  # bump après Step 1 (ex. 3)
EPOCHS = 200
IMGSZ = 1024
MODEL = "yolov8s.pt"  # ~11M params — PAS yolov8n
MIN_IMAGES_PER_CLASS = 300
MIN_BOXES_PER_CLASS = 500

In [ ]:
# Download + class-order guard + dataset health (Step 1)
from collections import Counter
from pathlib import Path

import yaml
from roboflow import Roboflow

dataset = (
    Roboflow(api_key=ROBOFLOW_API_KEY)
    .workspace(WORKSPACE)
    .project(PROJECT)
    .version(VERSION)
    .download("yolov8")
)
DATA_YAML = f"{dataset.location}/data.yaml"
ROOT = Path(dataset.location)

names = yaml.safe_load(open(DATA_YAML))["names"]
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]
print("Classes du dataset :", names)
attendu = ["acne", "blackheads", "dark_spots", "enlarged_pores", "wrinkles"]
assert [n.lower().replace("-", "_") for n in names] == attendu, (
    f"Ordre incorrect ! Attendu {attendu}, obtenu {names}."
)
print("Ordre des classes OK ✔")

box_counts: Counter = Counter()
img_with_class = {i: set() for i in range(len(attendu))}
for split in ("train", "valid", "test"):
    label_dir = ROOT / split / "labels"
    if not label_dir.is_dir():
        continue
    for label_path in label_dir.glob("*.txt"):
        for line in label_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            cls_id = int(float(line.split()[0]))
            box_counts[cls_id] += 1
            img_with_class[cls_id].add(label_path.stem)

print("\n=== Dataset health ===")
print(f"{'class':<16} {'images':>8} {'boxes':>8}  status")
weak = []
for i, name in enumerate(attendu):
    n_img, n_box = len(img_with_class[i]), box_counts[i]
    ok = n_img >= MIN_IMAGES_PER_CLASS and n_box >= MIN_BOXES_PER_CLASS
    status = "OK" if ok else "WEAK — fix/grow in Roboflow"
    if not ok:
        weak.append(name)
    print(f"{name:<16} {n_img:>8} {n_box:>8}  {status}")
if weak:
    print(f"\n⚠ Under target: {weak}")
    print("Universe: acne / blackheads / dark spots / enlarged pores / wrinkles")
else:
    print("\n✔ All classes meet targets.")

In [ ]:
# Train (Step 0 / 2) — assert yolov8s-scale params
from ultralytics import YOLO

model = YOLO(MODEL)
n_params = sum(p.numel() for p in model.model.parameters())
print(f"Modèle chargé : {MODEL} — {n_params:,} paramètres")
assert n_params > 8_000_000, (
    f"Attendu yolov8s (~11M), obtenu {n_params:,}. Pas de yolov8n."
)
print("Taille modèle OK ✔")

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,
    patience=50,
    cos_lr=True,
    close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, scale=0.5, fliplr=0.5,
    mixup=0.1, copy_paste=0.1,
)
BEST = str(results.save_dir / "weights" / "best.pt")
print("Meilleurs poids :", BEST)

In [ ]:
# Step 3 — artefacts d'évaluation
from pathlib import Path

save_dir = Path(results.save_dir)
print("=== Ouvrir ===")
for name in ("results.png", "confusion_matrix.png", "PR_curve.png", "results.csv"):
    path = save_dir / name
    print(f"  {'✔' if path.exists() else '·'} {path}")
print("Si mAP50 stagne ~0.15 → data (Step 1). Cible réaliste ~0.3–0.5, pas 0.8.")

In [ ]:
# Export ONNX + download (backend/models/best.onnx)
import shutil
from google.colab import files
from ultralytics import YOLO

onnx_path = YOLO(BEST).export(format="onnx", nms=True, imgsz=IMGSZ)
best_onnx = Path(results.save_dir) / "weights" / "best.onnx"
shutil.copy(onnx_path, best_onnx)
print("Modèle ONNX :", best_onnx)
files.download(str(best_onnx))

## Après le téléchargement

1. Remplacer `backend/models/best.onnx`.
2. Dans `backend/.env` :
   ```
   SKIN_YOLO_MODEL_PATH=models/best.onnx
   ANALYSIS_ENGINE=auto
   ```
3. Depuis `backend/` : `python -m train.verify_onnx`
4. Redémarrer le backend, lancer une analyse.
5. CPU trop lent à 1024 → retrain avec `IMGSZ = 768`.